# 04 · Capstone Story Kit
**Presentation & Data Storytelling with AI Tools** · Day 2 · Module 7

Everything you need to turn a dataset into slide-ready charts and audience-specific prompts:
1. **Pick a dataset**: one of three fictional practice datasets, or upload your own CSV.
2. **Get story charts**: sorted, highlighted, with action titles, ready for slides.
3. **Make a hero number** for your opening slide.
4. **Generate three audience prompts** (executive, operations, public) to paste into Claude, ChatGPT, Gemini or Gamma.
5. **Download** every chart as a PNG in one zip.

▶ Run each cell with **Shift + Enter**. Only change lines marked ✏️.

In [ ]:
# ▶ Tools, story style, chart helpers and the practice datasets
import pandas as pd
import numpy as np

# ▶ STORY STYLE — run once. Every chart after this cell gets the clean look.
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

NAVY, CORAL, GRAY, LIGHT, INK = "#1B2A41", "#FF6F59", "#BFC5CF", "#EEF1F5", "#4A5568"
plt.rcParams.update({
    "figure.figsize": (10, 5.6), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": LIGHT, "grid.linewidth": 1,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "axes.axisbelow": True,
    "xtick.color": INK, "ytick.color": INK, "ytick.left": False,
    "font.size": 11, "axes.titlesize": 16, "axes.titleweight": "bold",
})

def add_titles(ax, title, subtitle=""):
    """Headline (the insight) + a smaller subtitle (what is measured), left-aligned."""
    ax.set_title(title, loc="left", pad=34 if subtitle else 12, color=NAVY)
    if subtitle:
        ax.text(0, 1.035, subtitle, transform=ax.transAxes, fontsize=11, color=INK, va="bottom")

def add_source(fig, text):
    """Small source line in the bottom-left corner: always say where data came from."""
    fig.text(0.01, -0.02, text, fontsize=9, color="#8A94A6", ha="left", va="top")

print("Story style loaded ✔")

SAVED = []   # every chart you make is remembered here so you can download them all at the end

def _finish(fig, ax, title, subtitle, source):
    add_titles(ax, title, subtitle)
    add_source(fig, source)
    SAVED.append((title, fig))
    plt.show()

def story_line(series, highlight, title, subtitle="", source="", annotate=None):
    """series: {name: pandas Series indexed by date or number}. highlight: the name to color."""
    fig, ax = plt.subplots()
    for name, s in series.items():
        hero = name == highlight
        ax.plot(s.index, s.values, color=CORAL if hero else GRAY, lw=4 if hero else 1.6, zorder=3 if hero else 1)
        ax.text(s.index[-1], s.values[-1], "  " + str(name), color=CORAL if hero else INK,
                fontweight="bold" if hero else "normal", va="center", fontsize=10)
    if annotate:                                   # (x, y, "text")
        ax.annotate(annotate[2], xy=annotate[:2], xytext=(20, 30), textcoords="offset points",
                    arrowprops=dict(arrowstyle="->", color=INK), color=INK)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    _finish(fig, ax, title, subtitle, source)

def story_bar(values, highlight, title, subtitle="", source="", fmt="{:,.0f}"):
    """values: pandas Series (labels -> numbers). Sorted, horizontal, one highlighted bar."""
    s = values.sort_values()
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.45 * len(s) + 1.5)))
    ax.barh([str(i) for i in s.index], s.values, color=[CORAL if i == highlight else GRAY for i in s.index])
    for i, v in zip(s.index, s.values):
        ax.text(v, str(i), "  " + fmt.format(v), va="center", fontsize=10,
                color=CORAL if i == highlight else INK, fontweight="bold" if i == highlight else "normal")
    ax.grid(axis="y", visible=False); ax.grid(axis="x", color=LIGHT)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    _finish(fig, ax, title, subtitle, source)

def big_number(number_text, label, source=""):
    """A 'hero number' image for a stat slide."""
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.axis("off")
    ax.text(0.0, 0.62, number_text, fontsize=88, fontweight="bold", color=CORAL, va="center")
    ax.text(0.0, 0.18, label, fontsize=20, color=NAVY, va="center", wrap=True)
    add_source(fig, source)
    SAVED.append((label, fig))
    plt.show()

def make_er_data(seed=2025):
    """Fictional 'Riverside Community Hospital' emergency-room visits for 2025 (one row per visit)."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
    dow_mult = np.array([1.17, 1.04, 0.99, 0.97, 1.00, 0.90, 0.93])            # Mon ... Sun
    month_mult = np.array([1.14, 1.10, 1.00, 0.97, 0.98, 1.00, 1.03, 1.01, 0.98, 1.00, 1.04, 1.12])
    daily_n = rng.poisson(92 * dow_mult[dates.dayofweek] * month_mult[dates.month - 1])
    date = np.repeat(dates.values, daily_n)
    n = len(date)

    hour_w = np.array([2.0, 1.6, 1.3, 1.1, 1.0, 1.1, 1.6, 2.4, 3.4, 4.4, 5.0, 5.3,
                       5.3, 5.2, 5.1, 5.1, 5.2, 5.4, 5.6, 5.7, 5.3, 4.6, 3.7, 2.8])
    hour = rng.choice(24, size=n, p=hour_w / hour_w.sum())
    triage = rng.choice([1, 2, 3, 4, 5], size=n, p=[0.02, 0.13, 0.46, 0.31, 0.08])
    amb_p = np.array([0, 0.75, 0.45, 0.20, 0.06, 0.03])[triage]
    arrival = np.where(rng.random(n) < amb_p, "Ambulance", "Walk-in")
    age = rng.choice(["0-17", "18-44", "45-64", "65+"], size=n, p=[0.18, 0.37, 0.25, 0.20])

    crowd_day = np.repeat(daily_n / daily_n.mean(), daily_n)
    crowd_hour = hour_w[hour] / hour_w.mean()
    base_wait = np.array([0, 2, 12, 38, 52, 48])[triage]
    wait = base_wait * (0.55 + 0.45 * crowd_day) * (0.6 + 0.4 * crowd_hour) * rng.gamma(4, 0.25, n)
    wait = np.maximum(0, wait.round()).astype(int)

    lwbs_p = np.where(triage >= 3, np.clip(0.005 + 0.0012 * (wait - 30), 0.005, 0.40), 0.0)
    lwbs = rng.random(n) < lwbs_p
    admit_p = np.array([0, 0.80, 0.55, 0.25, 0.05, 0.01])[triage]
    disposition = np.where(lwbs, "Left without being seen",
                           np.where(rng.random(n) < admit_p, "Admitted", "Discharged"))
    treat = np.array([0, 300, 260, 210, 120, 80])[triage] * rng.gamma(3, 1 / 3, n)
    los = np.where(lwbs, wait, wait + treat.round()).astype(int)

    df = pd.DataFrame({
        "visit_id": np.arange(1, n + 1),
        "date": pd.to_datetime(date),
        "hour": hour,
        "age_group": age,
        "arrival_mode": arrival,
        "triage_level": triage,
        "wait_minutes": wait,
        "length_of_stay_minutes": los,
        "disposition": disposition,
    })
    df.insert(2, "weekday", df["date"].dt.day_name())
    df.insert(3, "month", df["date"].dt.month_name())
    return df

def make_rail_data(seed=7):
    """Fictional 'Potomac Valley Commuter Rail': average weekday riders by station and month, 2019-2025."""
    rng = np.random.default_rng(seed)
    stations = {"Union Terminal": ("North", 3200), "Harbor Point": ("North", 2100), "Crystal Ridge": ("North", 1900),
                "Old Town": ("North", 1500), "Springfield Junction": ("South", 1100), "Lakeview": ("South", 650),
                "Woodmont": ("South", 900), "Quarry Hill": ("South", 450), "Manor Park": ("West", 800),
                "Burke Hollow": ("West", 950), "Broad Meadow": ("West", 700)}
    months = pd.date_range("2019-01-01", "2025-12-01", freq="MS")
    t = np.arange(len(months))
    covid = np.ones(len(months))
    start = list(months).index(pd.Timestamp("2020-04-01"))
    covid[start - 1] = 0.6                                           # March 2020: the drop begins
    covid[start:] = np.clip(0.08 + 0.62 * (1 - np.exp(-(t[start:] - start) / 16)), 0, 1)
    rows = []
    for name, (line, base) in stations.items():
        recover = rng.uniform(0.85, 1.12)            # each station recovers a bit differently
        level = np.where(t < start - 1, 1.0, np.minimum(1.0, covid * recover))
        season = 1 + 0.05 * np.sin((months.month - 3) / 12 * 2 * np.pi)
        riders = base * level * season * rng.normal(1, 0.03, len(months))
        rows.append(pd.DataFrame({"month": months, "station": name, "line": line,
                                  "avg_weekday_riders": riders.round().astype(int)}))
    return pd.concat(rows, ignore_index=True)

def make_coffee_data(seed=11):
    """Fictional 'Ridgeline Coffee': weekly revenue by location and product category, 2024-2025."""
    rng = np.random.default_rng(seed)
    weeks = pd.date_range("2024-01-01", "2025-12-29", freq="W-MON")
    locs = {"Downtown": 1.35, "Campus": 1.10, "Riverside": 1.0, "Airport": 1.25, "Mall": 0.85, "Suburb": 0.75}
    cats = {"Hot coffee": 5200, "Cold brew": 1800, "Tea": 900, "Food": 2400}
    wk = np.arange(len(weeks))
    summer = np.sin((weeks.dayofyear - 80) / 365 * 2 * np.pi)          # +1 mid-summer, -1 mid-winter
    rows = []
    for loc, size in locs.items():
        campus_dip = np.where((loc == "Campus") & weeks.month.isin([6, 7, 8]), 0.55, 1.0)
        for cat, base in cats.items():
            growth = 1 + (0.45 if cat == "Cold brew" else 0.04) * wk / len(wk)
            seas = 1 + (0.55 if cat == "Cold brew" else -0.08 if cat == "Hot coffee" else 0.0) * summer
            revenue = base * size * growth * seas * campus_dip * rng.normal(1, 0.06, len(weeks))
            rows.append(pd.DataFrame({"week": weeks, "location": loc, "category": cat,
                                      "revenue_usd": revenue.round(2)}))
    return pd.concat(rows, ignore_index=True)

print("Kit loaded ✔  — next, pick your dataset")

## Step 1 · Pick your dataset ✏️

| `DATASET` | What it is (all practice data is **fictional**) |
| --- | --- |
| `"er"` | Riverside Community Hospital: 30,000+ emergency-room visits in 2025 (waits, triage, arrivals) |
| `"rail"` | Potomac Valley Commuter Rail: weekday riders at 11 stations, 2019–2025 (pandemic drop and recovery) |
| `"coffee"` | Ridgeline Coffee: weekly revenue at 6 cafés by product, 2024–2025 (cold brew boom, campus summer dip) |
| `"upload"` | Your own CSV file (you'll be asked to choose it) |

In [ ]:
DATASET = "rail"   # ✏️ "er", "rail", "coffee" or "upload"

if DATASET == "er":
    df = make_er_data()
elif DATASET == "rail":
    df = make_rail_data()
elif DATASET == "coffee":
    df = make_coffee_data()
elif DATASET == "upload":
    from google.colab import files
    uploaded = files.upload()                       # a file picker appears below this cell
    df = pd.read_csv(next(iter(uploaded)))
else:
    raise ValueError('DATASET must be "er", "rail", "coffee" or "upload"')

print(f"{DATASET}: {len(df):,} rows × {df.shape[1]} columns")
print("Columns:", ", ".join(df.columns))
df.head()

## Step 2 · Story charts for the practice datasets
Run this cell to get two ready-made story charts and the key facts for your dataset. (Uploaded your own data? Skip to **Step 2b**.)

In [ ]:
HERO = None
SRC = {"er": "Riverside Community Hospital ER visits, 2025 (FICTIONAL teaching data)",
       "rail": "Potomac Valley Commuter Rail ridership, 2019–2025 (FICTIONAL teaching data)",
       "coffee": "Ridgeline Coffee sales, 2024–2025 (FICTIONAL teaching data)"}.get(DATASET, "Your data")
FACTS = {}

if DATASET == "rail":
    total = df.groupby("month")["avg_weekday_riders"].sum()
    base19 = total.loc["2019"].mean(); now25 = total.loc["2025"].mean(); low = total.min()
    FACTS = {"2019 average weekday riders": f"{base19:,.0f}", "2025 average weekday riders": f"{now25:,.0f}",
             "2025 vs 2019": f"{now25 / base19:.0%}", "lowest month": f"{total.idxmin():%B %Y} ({low:,.0f} riders)"}
    HERO = (f"{now25 / base19:.0%}", "of 2019's weekday riders are back on the trains in 2025")
    by_line = df.groupby(["line", "month"])["avg_weekday_riders"].sum()
    story_line({ln: by_line[ln] for ln in ["North", "South", "West"]}, "North",
               f"Ridership has recovered to {now25 / base19:.0%} of its 2019 level",
               "Average weekday riders by line, monthly", "Source: " + SRC,
               annotate=(total.idxmin(), by_line["North"].min(), f"{total.idxmin():%B %Y} low"))
    rec = (df[df["month"].dt.year == 2025].groupby("station")["avg_weekday_riders"].mean()
           / df[df["month"].dt.year == 2019].groupby("station")["avg_weekday_riders"].mean() * 100)
    story_bar(rec, rec.idxmax(), f"{rec.idxmax()} has recovered the most: {rec.max():.0f}% of 2019 riders",
              "2025 average weekday riders as % of 2019, by station", "Source: " + SRC, fmt="{:.0f}%")

elif DATASET == "coffee":
    df["year"] = df["week"].dt.year
    cat = df.groupby(["category", "week"])["revenue_usd"].sum()
    cb = df[df["category"] == "Cold brew"].groupby("year")["revenue_usd"].sum()
    share25 = cb[2025] / df[df["year"] == 2025]["revenue_usd"].sum()
    FACTS = {"cold brew revenue growth 2024→2025": f"{cb[2025] / cb[2024] - 1:.0%}",
             "cold brew share of 2025 revenue": f"{share25:.0%}",
             "total 2025 revenue": f"${df[df['year'] == 2025]['revenue_usd'].sum():,.0f}"}
    HERO = (f"+{cb[2025] / cb[2024] - 1:.0%}", "growth in cold brew revenue from 2024 to 2025")
    story_line({c: cat[c].rolling(4).mean().dropna() for c in ["Hot coffee", "Cold brew", "Food", "Tea"]}, "Cold brew",
               f"Cold brew revenue grew {cb[2025] / cb[2024] - 1:.0%} in 2025 and now peaks every summer",
               "Weekly revenue by category, US$ (4-week average)", "Source: " + SRC)
    loc = df[df["year"] == 2025].groupby("location")["revenue_usd"].sum() / 1000
    camp = df[df["location"] == "Campus"].groupby("week")["revenue_usd"].sum()
    dip = 1 - camp[camp.index.month.isin([6, 7, 8])].mean() / camp[~camp.index.month.isin([6, 7, 8])].mean()
    rank = int(loc.rank(ascending=False)["Campus"])
    FACTS["Campus summer sales drop"] = f"{dip:.0%}"
    story_bar(loc, "Campus", f"Campus ranks #{rank} of {len(loc)} in 2025 revenue despite a {dip:.0%} summer drop",
              "2025 revenue by location, US$ thousands", "Source: " + SRC, fmt="${:,.0f}k")

elif DATASET == "er":
    vpd = df.groupby("weekday").size() / df.groupby("weekday")["date"].nunique()
    wait_m = df.groupby(df["date"].dt.month)["wait_minutes"].median()
    FACTS = {"visits per day (average)": f"{len(df) / df['date'].nunique():.0f}",
             "busiest weekday": f"{vpd.idxmax()} ({vpd.max():.0f} visits/day)",
             "median wait": f"{df['wait_minutes'].median():.0f} minutes",
             "left without being seen": f"{(df['disposition'] == 'Left without being seen').mean():.1%}"}
    HERO = (f"{df['wait_minutes'].median():.0f} min", "median wait to see a clinician at Riverside's ER in 2025")
    story_bar(vpd.round(0), vpd.idxmax(), f"{vpd.idxmax()}s are the busiest day: {vpd.max():.0f} visits vs. {vpd.min():.0f} on {vpd.idxmin()}s",
              "Average ER visits per day, by weekday, 2025", "Source: " + SRC)
    wait_m.index = pd.to_datetime([f"2025-{m:02d}-01" for m in wait_m.index])
    story_line({"Median wait": wait_m}, "Median wait",
               f"Median waits peak in winter at {wait_m.max():.0f} minutes",
               "Median wait to see a clinician (minutes), by month, 2025", "Source: " + SRC)
else:
    print("Uploaded data: use Step 2b to build your own chart.")

pd.Series(FACTS, name="key facts (verified by code)").to_frame()

## Step 2b · Build your own chart (works with any dataset) ✏️
Fill in the column names from the list printed in Step 1. `AGG` decides how rows are combined.

In [ ]:
# Defaults below work for "rail". For "er" try: X_COL="age_group", Y_COL="wait_minutes", AGG="median", HIGHLIGHT="65+"
# For "coffee" try: X_COL="location", Y_COL="revenue_usd", AGG="sum", HIGHLIGHT="Downtown"
CHART      = "bar"          # ✏️ "bar" (compare categories) or "line" (change over time)
X_COL      = "station"      # ✏️ categories (bar) or dates/numbers (line)
Y_COL      = "avg_weekday_riders"   # ✏️ the number to show
AGG        = "mean"         # ✏️ "sum", "mean", "median" or "count"
HIGHLIGHT  = "Old Town"     # ✏️ the one category to color (bar), or None
TITLE      = "Write your insight here, with a number"      # ✏️ an action title
SUBTITLE   = "What is measured, units, time period"        # ✏️
SOURCE     = "Source: " + SRC                               # ✏️

missing = [c for c in (X_COL, Y_COL) if c not in df.columns]
if missing:
    raise ValueError(f"Column(s) {missing} not found. Your columns are: {', '.join(df.columns)}")
data = df.copy()
if CHART == "line":
    if data[X_COL].dtype == object:
        try:
            data[X_COL] = pd.to_datetime(data[X_COL])
        except (ValueError, TypeError):
            pass                                    # not dates: keep as they are
    s = data.groupby(X_COL)[Y_COL].agg(AGG).sort_index()
    story_line({Y_COL: s}, Y_COL, TITLE, SUBTITLE, SOURCE)
else:
    s = data.groupby(X_COL)[Y_COL].agg(AGG)
    story_bar(s, HIGHLIGHT, TITLE, SUBTITLE, SOURCE)

## Step 3 · A hero number for your opening slide ✏️
One big number with one plain sentence is often the strongest opening slide.

In [ ]:
NUMBER = HERO[0] if HERO else "1 in 4"                     # ✏️ or type your own number
LABEL  = HERO[1] if HERO else "say what the number means"   # ✏️ one plain sentence
big_number(str(NUMBER), str(LABEL), "Source: " + SRC)

## Step 4 · Same data, three audiences: prompts to copy
Each prompt carries your **verified** facts, so the AI writes the story without inventing numbers. Copy one into Claude, ChatGPT, Gemini, Gamma or Copilot in PowerPoint.

In [ ]:
TOPIC = {"er": "emergency-room waits at Riverside Community Hospital",
         "rail": "ridership recovery on Potomac Valley Commuter Rail",
         "coffee": "the cold brew boom at Ridgeline Coffee"}.get(DATASET, "my dataset")   # ✏️
facts_text = "\n".join(f"- {k}: {v}" for k, v in FACTS.items()) or "- (paste your verified facts here)"

audiences = {
    "EXECUTIVE": ("a time-pressed executive deciding on budget", "5 slides: headline number, why it matters, 1 chart, 2 options with cost/benefit, the ask", "confident, brief"),
    "OPERATIONS": ("the operations team who will change schedules and processes", "6 slides: what we found, where and when it happens, 2 charts, root causes to check, 3 concrete next steps with owners", "practical, specific"),
    "PUBLIC": ("community members with no technical background", "4 slides: a relatable opening, one simple chart, what it means for you, what's being done", "warm, plain language, no jargon"),
}
for name, (aud, fmt, tone) in audiences.items():
    print(f"===== {name} PROMPT =====")
    print(f"""ROLE: You are an expert presentation designer and data storyteller.
AUDIENCE: {aud}.
TASK: Draft a slide outline about {TOPIC}.
FORMAT: {fmt}. For each slide give an action title (a full sentence with the key number), 2-3 bullets, and a visual suggestion.
TONE: {tone}.
CONSTRAINTS: Use ONLY these verified facts. Do not add numbers. Mark anything you assume with [ASSUMPTION].
{facts_text}
EVALUATION: Before answering, score your draft 1-5 on clarity for this audience and fix anything below 4.
""")

## Step 5 · Download all your charts
Saves every chart from this session as a high-resolution PNG and downloads them in one zip.

In [ ]:
import os, re, zipfile
os.makedirs("capstone_charts", exist_ok=True)
paths = []
for i, (title, fig) in enumerate(SAVED, start=1):
    name = f"capstone_charts/{i:02d}_" + re.sub(r"[^a-zA-Z0-9]+", "_", title)[:50].strip("_") + ".png"
    fig.savefig(name, dpi=200, bbox_inches="tight")
    paths.append(name)
with zipfile.ZipFile("capstone_charts.zip", "w") as z:
    for p in paths:
        z.write(p)
print(f"{len(paths)} charts saved")
try:
    from google.colab import files
    files.download("capstone_charts.zip")
except ImportError:
    print("Saved capstone_charts.zip")

## Capstone checklist
- [ ] One-sentence Big Idea, written before the slides.
- [ ] Every number on a slide traces back to a code cell or a cited source.
- [ ] Action titles: each slide title is a full sentence with the finding.
- [ ] Each chart passes the honest-chart checklist (Notebook 02).
- [ ] The deck fits **one** audience; you can name what that audience should do next.
- [ ] AI help is disclosed (e.g., "Drafted with Claude; numbers verified in Python").